# Frozen stability_v1 — supervised Colab GPU migration (D-020)
Run cells in order. Setup/readiness do **not** launch primary training. Use only this notebook; do not use aanet files.

Select **Runtime → Change runtime type → GPU**. Upload the senior-reviewed `colab_transfer.zip` and `TRANSFER_RECEIPT.json` to `MyDrive/DeepAA_migration` before proceeding. Draft packages are intentionally blocked. No cloud project or payment is required by this notebook.

The remaining primary queue is fresh K5/seed503, then K7 seeds113/211/307/401/503. All frozen settings remain unchanged. Mixed CPU/GPU execution is a limitation, not a numerical-equivalence claim.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, hashlib, subprocess, sys, zipfile, os
sys.dont_write_bytecode = True
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'

drive.mount('/content/drive')  # Normal user authorization; no credentials in package.
UPLOAD = Path('/content/drive/MyDrive/DeepAA_migration')
DRIVE_DATA = Path('/content/drive/MyDrive/HAM10000')  # Candidate, verified later.
DURABLE = Path('/content/drive/MyDrive/DeepAA_stability_v1_D020')  # Keep fixed across sessions.
PAYLOAD = Path('/content/deepaa_transfer')
WORKSPACE = Path('/content/deepaa_workspace')
LOCAL_DATA = Path('/content/HAM10000')
ENV = Path('/content/deepaa_env')
PRIMARY_LAUNCH_ENABLED = False
print('Primary launch disabled. Dataset location is not yet verified.')


## Verify the transfer before executing packaged code
The senior receipt is the trust anchor. Compare its ZIP SHA-256 with the senior's review output. ZIP extraction rejects unsafe names and verifies every file. No download from the public repository is used.


In [ ]:
receipt = json.loads((UPLOAD/'TRANSFER_RECEIPT.json').read_text())
assert receipt['release'], 'STOP: disabled draft. Senior must commit/review and rebuild release package.'
archive = UPLOAD/receipt['archive']
with archive.open('rb') as f: actual_zip = hashlib.file_digest(f, 'sha256').hexdigest()
assert actual_zip == receipt['sha256'], 'Transfer SHA-256 mismatch'
assert not PAYLOAD.exists(), 'Extraction exists. Do not merge. Review or start a fresh runtime.'
with zipfile.ZipFile(archive) as z:
    names = z.namelist()
    assert len(names) == len(set(names))
    for name in names:
        assert z.getinfo(name).orig_filename == name, 'Unsafe normalized ZIP member'
        p = Path(name)
        assert not p.is_absolute() and '..' not in p.parts and '\\' not in name and ':' not in name
        assert ((z.getinfo(name).external_attr >> 16) & 0o170000) != 0o120000
    z.extractall(PAYLOAD)
package = json.loads((PAYLOAD/'PACKAGE.json').read_text())
def file_sha(p):
    with p.open('rb') as f: return hashlib.file_digest(f, 'sha256').hexdigest()
assert file_sha(PAYLOAD/'PACKAGE.json') == receipt['package_manifest_sha256']
actual = {p.relative_to(PAYLOAD).as_posix(): {'bytes': p.stat().st_size, 'sha256': file_sha(p)} for p in PAYLOAD.rglob('*') if p.is_file() and p != PAYLOAD/'PACKAGE.json'}
assert actual == package['files']
assert package['release'] and package['head'] == receipt['head'] and not package['source_status']
print(json.dumps(receipt, indent=2))
sys.path.insert(0, str(PAYLOAD/'source_bytes'))
from research.code.colab_transfer import restore, run_visible
ROOT = restore(PAYLOAD, WORKSPACE)
print('Restored clean reviewed source with exact packaged bytes and verified D-019 receipt:', ROOT)


## Isolated pinned environment — stop on any incompatibility
This downloads Python 3.13.3 and the explicit candidate Linux packages. It does not use Colab's TensorFlow defaults. If a wheel/version or GPU driver is incompatible, **stop and paste the error**; do not downgrade or alter settings. GPU compatibility is not yet established.


In [ ]:
assert not ENV.exists(), 'Environment already exists; do not silently update it.'
run_visible([sys.executable, '-m', 'pip', 'install', 'uv==0.8.22'])
run_visible([sys.executable, '-m', 'uv', 'venv', '--python', '3.13.3', '--seed', str(ENV)])
PYTHON = str(ENV/'bin/python')
INSTALL_REPORT = WORKSPACE/'installation.json'
run_visible([PYTHON, '-m', 'pip', 'install', '--ignore-installed', '--report', str(INSTALL_REPORT), '-r', str(ROOT/'research/requirements_colab.txt')])
run_visible([PYTHON, '-m', 'pip', 'check'])
resolved = json.loads(INSTALL_REPORT.read_text())
lines = []
for item in resolved['install']:
    digest = item['download_info']['archive_info']['hashes']['sha256']
    lines.append(item['metadata']['name']+'=='+item['metadata']['version']+' --hash=sha256:'+digest)
(WORKSPACE/'requirements_resolved_linux.txt').write_text('\n'.join(sorted(lines))+'\n')
print('Target-specific exact wheel-hash lock:', WORKSPACE/'requirements_resolved_linux.txt')


## Verify Drive data and copy to runtime-local storage
Hashes all 10,015 JPEGs and metadata against the Windows-generated source inventory. Frozen split bytes determine membership/order. No dataset upload, split regeneration or Drive-original mutation occurs.


In [ ]:
from research.code.colab_runtime import prepare_images
spec = json.loads((PAYLOAD/'DATASET.json').read_text())
prepare_images(DRIVE_DATA, LOCAL_DATA, spec)
print('Drive and runtime-local image/metadata bytes verified:', len(spec['files']))


## Local orchestration fixtures on the target (no model training)
These tests check preservation, paths, partial-attempt refusal and existing workflow safety. GPU checks run only in the next cell.


In [ ]:
run_visible([PYTHON, '-B', '-m', 'pytest', 'research/tests/test_colab_migration.py', 'research/tests/test_workflow.py', 'research/tests/test_output_contract.py', 'research/tests/test_degenerate_recovery.py', '-q', '-p', 'no:cacheprovider'], cwd=ROOT, env=dict(os.environ, PYTHONDONTWRITEBYTECODE='1'))


## Readiness: GPU, deterministic training-only benchmarks, old evidence
This cell performs **non-scientific, training-membership-only** GPU steps (K5 and K7 twice) and zero-optimizer artifact validation of existing work. It never starts a primary attempt. Timings exclude validation/export/persistence and cannot guarantee Colab lifetime. A failed historical receipt, relocation or numerical check blocks primary launch.

Do not start readiness in parallel with training. Disconnect/delete any previous project runtime before using a new one.


In [ ]:
BASE = [PYTHON, '-B', '-u', str(ROOT/'research/tools/colab_migration.py')]
COMMON = ['--payload', str(PAYLOAD), '--workspace', str(WORKSPACE), '--durable', str(DURABLE), '--dataset', str(LOCAL_DATA)]
READY_STDOUT = WORKSPACE/'readiness_stdout.txt'
READY_RESULT = WORKSPACE/'readiness_result.json'
assert not READY_STDOUT.exists() and not READY_RESULT.exists(), 'Readiness output exists; review before repeating.'
run_visible(BASE+['readiness']+COMMON+['--result-output', str(READY_RESULT)], log=READY_STDOUT)
# Progress and failures are visible; machine-readable JSON has its own file.
ready_output = json.loads(READY_RESULT.read_text())
READINESS = Path(ready_output['path'])
READINESS_SHA256 = ready_output['sha256']
from research.code.colab_runtime import save_snapshot
save_snapshot(WORKSPACE/'readiness', DURABLE/'readiness_files')
# Installation records and target wheel lock are also durable, without credentials.
from research.code.colab_transfer import copy_new
setup_target = DURABLE/'environment_records'/READINESS.parent.name
setup_target.mkdir(parents=True, exist_ok=False)
for name in ('installation.json', 'requirements_resolved_linux.txt'):
    copy_new(WORKSPACE/name, setup_target/name)
print('READINESS_PATH =', READINESS)
print('READINESS_SHA256 =', READINESS_SHA256)
print(json.dumps(ready_output, indent=2))


## Stop for senior review
Paste back the complete READINESS output and SHA-256, transfer receipt, GPU/driver/CUDA/cuDNN details, package freeze, K5/K7 repeat fingerprints, timing/memory and any failed cell/log. Do not edit tolerances or skip failures.

GPU primary launch remains blocked until the senior approves this exact readiness SHA-256. Colab may disconnect at any time. Workflow continuation is not model-training resume. Drive records do not provide atomic cross-runtime locking; the procedure depends on the explicit single-runtime discipline you accepted.


In [ ]:
# PRIMARY TRAINING — MANUAL SENIOR LAUNCH ONLY. Defaults disabled.
PRIMARY_LAUNCH_ENABLED = False
REVIEWED_READINESS_SHA256 = ''  # Senior pastes the approved digest after review.
SINGLE_RUNTIME_CONFIRMATION = ''  # Type the exact sentence below only after doing it.
# I deleted or disconnected all previous project runtimes and will use only this project session.

if not PRIMARY_LAUNCH_ENABLED:
    print('PRIMARY TRAINING DISABLED. No launch performed.')
else:
    from research.code.colab_runtime import CONFIRMATION
    assert SINGLE_RUNTIME_CONFIRMATION == CONFIRMATION
    assert REVIEWED_READINESS_SHA256 == READINESS_SHA256 == file_sha(READINESS)
    run_visible(BASE+['launch']+COMMON+['--readiness', str(READINESS), '--reviewed-readiness-sha256', REVIEWED_READINESS_SHA256, '--single-runtime-confirmation', SINGLE_RUNTIME_CONFIRMATION])


## After success or interruption
Completed attempts are durably hash-verified and archived before another primary run. Preserve all Drive records. If `QUEUE_ACTIVE.json` or an unresolved registration blocks execution, stop for senior review; do not delete records, take over, rerun or create a different project directory to bypass them. A runtime loss may lose the last unpersisted bytes. No automatic cleanup/retry or keep-alive is provided.

Return the final bundle's `ATTEMPT_AUDIT.json`, numerical metrics and `BUNDLE_SHA256.json`. Both seed503 attempts remain in the audit; the scientific identity counts once.
